# Processing Frog Spectrograms

Measures acoustic parameters from the saved frog oscillogram images: element
length, inter-element interval, inter-burst interval, and elements per burst
(median, min, max).

| Output Column | Description |
|---|---|
| `Element_Length` | Mean element (pulse) duration in seconds |
| `Inter-Element_Interval` | Mean within-burst gap in seconds |
| `Inter-Burst_Interval` | Mean between-burst gap in seconds; 0 for species with no burst structure |
| `Elements_Per_Burst` | Median elements per burst; 1 for non-burst species |
| `Min_Elements_Per_Burst` | Fewest elements found in any one burst; 1 for non-burst species |
| `Max_Elements_Per_Burst` | Most elements found in any one burst; 1 for non-burst species |

Same detection approach as the cricket and katydid pipelines, minus the OCR
step--`pixel_time` comes straight from the clip log instead of being read off
the image.

**Run order:** `Frog_Clip_Log.ipynb` (produces `clips_ready`) → this notebook

In [ ]:
import csv
import os
import re
import statistics
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

# Cropped clips in, generated oscillogram PNGs out.
BASE_DIR = Path.home() / 'Discrete_Signals'
AUDIO_DIR = BASE_DIR / 'Cropped_Frogs_Audios'
SPEC_DIR = BASE_DIR / 'Cropped_Frogs_Specs'
SPEC_DIR.mkdir(parents=True, exist_ok=True)

## Constants

In [ ]:
# Height fraction skipped around the zero-amplitude baseline when measuring
# ink--small, so quiet elements near the baseline still register.
CENTER_EXCLUDE_FRACTION = 0.03

# Oscillogram DPI; fixed so pixel_time = clip_duration / image_width holds.
SAVE_DPI = 150


## Load Clip Log

Reads `frog_clip_log.csv` directly rather than via `%store`. Keeps only the
`ok` rows and matches each one to its wav file on disk.

In [ ]:
clips_ready = pd.read_csv(AUDIO_DIR / 'frog_clip_log.csv')

for col in ('audio_num', 'source_dur_s', 'start_time_s', 'end_time_s', 'clip_duration_s'):
    clips_ready[col] = pd.to_numeric(clips_ready[col], errors='coerce')

clips_ready = clips_ready[clips_ready['status'] == 'ok'].copy()

# Match each row to its actual wav file (flat in AUDIO_DIR, not the recorded
# container path in the CSV, which no longer resolves on this machine).
clips_ready['wav_name'] = clips_ready.apply(
    lambda r: f"{r['genus']}_{r['species']}_{int(r['audio_num'])}.wav", axis=1
)
clips_ready['clipped_file'] = clips_ready['wav_name'].apply(lambda n: str(AUDIO_DIR / n))
clips_ready = clips_ready[clips_ready['clipped_file'].apply(lambda p: Path(p).exists())]
clips_ready = clips_ready.reset_index(drop=True)

print(f'{len(clips_ready)} clips loaded')
clips_ready.head()


## Helper Functions

Same as the cricket and katydid pipelines.

In [ ]:
def gaussian_filter1d(signal, sigma):
    """Smooth a 1-D array with a Gaussian kernel."""
    kernel_radius = int(4 * sigma + 0.5)
    kernel_positions = np.arange(-kernel_radius, kernel_radius + 1, dtype=float)
    kernel = np.exp(-0.5 * (kernel_positions / sigma) ** 2)
    kernel /= kernel.sum()
    return np.convolve(signal, kernel, mode='same')


def silhouette(values, cluster_labels):
    """Mean silhouette score for a 1-D binary clustering."""
    total_score = 0.0
    for i, value in enumerate(values):
        same_cluster = values[cluster_labels == cluster_labels[i]]
        other_cluster = values[cluster_labels != cluster_labels[i]]
        within_dist = np.mean(np.abs(same_cluster - value)) if len(same_cluster) > 1 else 0.0
        between_dist = np.mean(np.abs(other_cluster - value)) if len(other_cluster) > 0 else 0.0
        denom = max(within_dist, between_dist)
        total_score += (between_dist - within_dist) / denom if denom > 0 else 0.0
    return total_score / len(values)


def kmeans2(values):
    """Best 2-way split of a 1-D array by exhaustive search over split points.
    Returns (cluster_labels, cluster_centers); label 0 = short, 1 = long."""
    sorted_values = np.sort(values)
    best_silhouette = -2.0
    best_split_index = 1

    for i in range(1, len(sorted_values)):
        if i > 1 and sorted_values[i] == sorted_values[i - 1]:
            continue
        split_threshold = (sorted_values[i - 1] + sorted_values[i]) / 2
        cluster_labels = (values > split_threshold).astype(int)
        if len(set(cluster_labels)) < 2:
            continue
        split_silhouette = silhouette(values, cluster_labels)
        if split_silhouette > best_silhouette:
            best_silhouette = split_silhouette
            best_split_index = i

    split_threshold = (sorted_values[best_split_index - 1] + sorted_values[best_split_index]) / 2
    cluster_labels = (values > split_threshold).astype(int)
    cluster_centers = np.array([
        values[cluster_labels == 0].mean() if (cluster_labels == 0).any() else sorted_values[0],
        values[cluster_labels == 1].mean() if (cluster_labels == 1).any() else sorted_values[-1],
    ])
    return cluster_labels, cluster_centers


def choose_k(gap_durations):
    """
    Return 2 if the gaps split cleanly into a short-gap and long-gap cluster
    (silhouette > 0.3), otherwise 1 (no real burst structure).
    """
    gap_durations = np.asarray(gap_durations, dtype=float)
    if len(gap_durations) < 3 or len(np.unique(gap_durations)) < 2:
        return 1
    cluster_labels, _ = kmeans2(gap_durations)
    if len(set(cluster_labels)) < 2:
        return 1
    return 2 if silhouette(gap_durations, cluster_labels) > 0.3 else 1


def rle(signal_list):
    """Run-length encode a 1-D sequence."""
    runs = []
    current_value = signal_list[0]
    run_length = 1
    for value in signal_list[1:]:
        if value == current_value:
            run_length += 1
        else:
            runs.append((current_value, run_length))
            current_value = value
            run_length = 1
    runs.append((current_value, run_length))
    return runs


## Signal Analysis

Same approach as the katydid pipeline, tuned for frogs: gentler smoothing, no
gap-filling (frog inter-element gaps are real, not noise), and ink measured
around a thin band near the detected centerline rather than a wide one.

In [ ]:
def clean_signal_runs(signal_list, pixel_time, fill_gap_size=0):
    """Drop on-runs shorter than 3 ms (noise); optionally fill short off-gaps
    that sit between two on-runs."""
    signal_list = np.asarray(signal_list, dtype=np.uint8)
    min_on_pixels = max(1, round(0.003 / pixel_time))

    cleaned = []
    for value, run_length in rle(signal_list):
        if value == 1 and run_length < min_on_pixels:
            cleaned.extend([0] * run_length)
        else:
            cleaned.extend([int(value)] * run_length)
    cleaned = np.array(cleaned, dtype=np.uint8)

    if fill_gap_size > 0 and len(cleaned) > 2:
        filled = []
        runs = rle(cleaned)
        for i, (value, run_length) in enumerate(runs):
            surrounded_by_signal = (
                i > 0 and i < len(runs) - 1
                and runs[i - 1][0] == 1 and runs[i + 1][0] == 1
            )
            if value == 0 and surrounded_by_signal and run_length <= fill_gap_size:
                filled.extend([1] * run_length)
            else:
                filled.extend([int(value)] * run_length)
        cleaned = np.array(filled, dtype=np.uint8)

    return cleaned


def find_centerline_row(spec_array):
    """Row of the always-dark zero-amplitude baseline, found by darkness rather
    than assumed to be the image's middle (a tight bbox can crop asymmetrically)."""
    row_dark_fraction = (spec_array < 200).mean(axis=1)
    return int(np.argmax(row_dark_fraction))


def extract_outer_band_ink(spec_array):
    """2-D ink array with only a thin band around the detected centerline
    excluded, so quiet near-baseline elements still register."""
    img_height, img_width = spec_array.shape
    center_row = find_centerline_row(spec_array)
    half_exclude = max(1, int(img_height * CENTER_EXCLUDE_FRACTION / 2))
    rows = np.arange(img_height)
    keep_mask = np.abs(rows - center_row) > half_exclude
    band = spec_array[keep_mask, :].astype(float)

    # Ink = darkness relative to background (white background -> near-zero ink in gaps)
    background_level = np.percentile(band, 95)
    ink = np.clip(background_level - band, 0, None)
    ink_peak = np.percentile(ink, 99)
    if ink_peak > 0:
        ink /= ink_peak

    return ink


def detect_signal_list_adaptive(ink_array, pixel_time):
    """Threshold the ink array into a binary on/off signal via adaptive
    hysteresis. Walks a standard threshold ladder, then an extended
    high-baseline one, then falls back to a single continuous on-run; see
    . Returns (signal_list, normalized_signal)."""
    col_pct85 = np.percentile(ink_array, 85, axis=0)
    col_pct95 = np.percentile(ink_array, 95, axis=0)
    col_pct99 = np.percentile(ink_array, 99, axis=0)
    col_signal_strength = gaussian_filter1d(
        0.20 * col_pct85 + 0.35 * col_pct95 + 0.45 * col_pct99,
        sigma=0.45
    )

    signal_floor = np.percentile(col_signal_strength, 5)
    signal_peak = np.percentile(col_signal_strength, 99.5)
    if signal_peak <= signal_floor:
        raise ValueError('No contrast in ink array--image may be blank')
    normalized_signal = np.clip(
        (col_signal_strength - signal_floor) / (signal_peak - signal_floor), 0, 1
    )

    min_on_pixels = max(1, round(0.003 / pixel_time))

    def try_ladder(ladder):
        candidates = []
        for high_threshold, low_threshold in ladder:
            high_mask = normalized_signal >= high_threshold
            low_mask = normalized_signal >= low_threshold

            signal_list = np.zeros_like(low_mask, dtype=np.uint8)
            column_pos = 0
            for value, run_length in rle(low_mask.astype(np.uint8)):
                run_start, run_end = column_pos, column_pos + run_length
                if value == 1 and run_length >= min_on_pixels and np.any(high_mask[run_start:run_end]):
                    signal_list[run_start:run_end] = 1
                column_pos = run_end

            signal_list = clean_signal_runs(signal_list, pixel_time, fill_gap_size=0)
            active_fraction = float(signal_list.mean())
            if active_fraction <= 0 or active_fraction >= 0.90:
                continue

            on_run_lengths = [run_length for value, run_length in rle(signal_list) if value == 1]
            if not on_run_lengths:
                continue

            median_on_length = float(np.median(on_run_lengths))
            tiny_run_fraction = sum(length <= 1 for length in on_run_lengths) / len(on_run_lengths)
            score = high_threshold - 0.15 * tiny_run_fraction - 0.002 * median_on_length
            candidates.append((score, signal_list))
        return candidates

    standard_ladder = [(t, max(0.06, t * 0.45)) for t in
                        [0.78, 0.70, 0.62, 0.54, 0.46, 0.38, 0.30, 0.22, 0.16, 0.10]]
    candidate_thresholds = try_ladder(standard_ladder)

    if not candidate_thresholds:
        extended_ladder = [(t, max(0.06, t - 0.12)) for t in [0.98, 0.94, 0.90, 0.86, 0.82]]
        candidate_thresholds = try_ladder(extended_ladder)

    if not candidate_thresholds:
        # Both ladders empty means a genuinely continuous trill, not a
        # detection failure.
        low_threshold = 0.10
        low_mask = normalized_signal >= low_threshold
        signal_list = clean_signal_runs(low_mask.astype(np.uint8), pixel_time, fill_gap_size=0)
        if signal_list.mean() <= 0:
            raise ValueError('No valid signal detected at any threshold')
        return signal_list, normalized_signal

    candidate_thresholds.sort(key=lambda t: t[0], reverse=True)
    return candidate_thresholds[0][1], normalized_signal


## Interval Classification

Same logic as the cricket pipeline--frog burst gaps are large and obvious, so
there's no need to raise the threshold the way the katydid pipeline does.

In [ ]:
def classify_intervals(time_bucket_list, leading_silence, trailing_silence,
                       element_length, pixel_time):
    """Classify off-durations as inter-element vs. inter-burst by clustering
    them into a short and long group and checking the split is real burst
    structure (guards explained in ).

    Returns (inter_element_interval, inter_burst_interval, elements_per_burst,
    min_elements_per_burst, max_elements_per_burst); no burst structure gives
    (mean gap, 0, 1, 1, 1).
    """
    off_durations = np.array(
        [duration for state, duration in time_bucket_list if state == 'Off'],
        dtype=float,
    )

    if len(off_durations) == 0:
        edge_gaps = [x for x in [leading_silence, trailing_silence] if x > 0]
        return 0.0, float(np.mean(edge_gaps)) if edge_gaps else 0.0, 1, 1, 1

    if len(off_durations) < 4 or len(np.unique(np.round(off_durations, 6))) < 2:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1

    if choose_k(off_durations) != 2:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1

    cluster_labels, cluster_centers = kmeans2(off_durations)
    short_gap_durations = off_durations[cluster_labels == 0]
    long_gap_durations = off_durations[cluster_labels == 1]
    inter_element_mean = float(np.mean(short_gap_durations))
    inter_burst_mean = float(np.mean(long_gap_durations))

    if inter_element_mean <= 0:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1

    burst_gap_ratio = inter_burst_mean / inter_element_mean
    absolute_separation = inter_burst_mean - inter_element_mean
    long_gap_fraction = len(long_gap_durations) / len(off_durations)
    n_long_gaps = len(long_gap_durations)
    inter_burst_pixels = inter_burst_mean / pixel_time

    if n_long_gaps == 1:
        single_long_gap_index = int(np.where(cluster_labels == 1)[0][0])
        single_gap_position = single_long_gap_index / max(1, len(off_durations) - 1)
    else:
        single_gap_position = 0.5

    is_burst = (
        burst_gap_ratio >= 4.0
        and inter_burst_mean >= 2.5 * element_length
        and absolute_separation >= 1.5 * element_length
        and long_gap_fraction <= 0.55
        and inter_burst_pixels >= 10
        and (n_long_gaps >= 2 or (inter_burst_pixels >= 200 and single_gap_position >= 0.25))
    )

    if not is_burst:
        return float(np.mean(off_durations)), 0.0, 1, 1, 1

    burst_boundary_time = (inter_element_mean + inter_burst_mean) / 2
    elements_per_burst_list = []
    current_count = 0
    for state, duration in time_bucket_list:
        if state == 'On':
            current_count += 1
        elif duration > burst_boundary_time:
            if current_count > 0:
                elements_per_burst_list.append(current_count)
            current_count = 0
    if current_count > 0:
        elements_per_burst_list.append(current_count)

    median_elements_per_burst = (
        int(statistics.median(elements_per_burst_list)) if elements_per_burst_list else 1
    )
    min_elements_per_burst = min(elements_per_burst_list) if elements_per_burst_list else 1
    max_elements_per_burst = max(elements_per_burst_list) if elements_per_burst_list else 1
    return (
        inter_element_mean, inter_burst_mean, max(1, median_elements_per_burst),
        min_elements_per_burst, max_elements_per_burst,
    )


## Spectrogram Image Generation

Turns each cropped WAV file into an oscillogram PNG: load the audio, bandpass
filter around its dominant frequency to cut background noise, and save a
filled waveform with no axes.

In [ ]:
def find_dominant_frequency(signal, sample_rate):
    """Dominant frequency of the clip above 500 Hz (the floor rejects mains hum
    and low-frequency environmental noise)."""
    spectrum = np.abs(np.fft.rfft(signal))
    freqs = np.fft.rfftfreq(len(signal), d=1 / sample_rate)
    above_500 = freqs > 500
    peak_idx = np.argmax(spectrum[above_500])
    return freqs[above_500][peak_idx]


def generate_spectrogram_image(wav_path, output_path, figsize=(10, 2)):
    """Bandpass-filter a cropped WAV clip around its dominant frequency and save
    a filled, axis-free oscillogram PNG (width = figsize[0] * SAVE_DPI)."""
    signal, sample_rate = librosa.load(str(wav_path), sr=None)
    duration_seconds = len(signal) / sample_rate

    dominant_freq = find_dominant_frequency(signal, sample_rate)
    band_width_hz = 500  # Hz on each side of the dominant frequency

    stft_matrix = librosa.stft(signal)
    magnitude = np.abs(stft_matrix)
    phase = np.angle(stft_matrix)
    frequencies = librosa.fft_frequencies(sr=sample_rate)

    freq_mask = (
        (frequencies >= dominant_freq - band_width_hz) &
        (frequencies <= dominant_freq + band_width_hz)
    )
    magnitude_filtered = magnitude * freq_mask[:, np.newaxis]

    # Suppress the quietest 80% of the filtered magnitude to remove residual noise.
    noise_floor = np.percentile(magnitude_filtered[magnitude_filtered > 0], 80)
    magnitude_filtered[magnitude_filtered < noise_floor] = 0

    filtered_signal = librosa.istft(magnitude_filtered * np.exp(1j * phase))

    time_axis = np.linspace(0, len(filtered_signal) / sample_rate, len(filtered_signal))

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(time_axis, filtered_signal, color='black', linewidth=0.5)
    ax.fill_between(time_axis, filtered_signal, alpha=1.0, color='black')
    # Pin the x-axis to [0, duration]; otherwise matplotlib's autoscale margin
    # is baked into the PNG and skews every downstream pixel_time.
    ax.set_xlim(0, duration_seconds)
    ax.axis('off')
    plt.tight_layout(pad=0)

    fig.savefig(str(output_path), dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0)
    plt.close(fig)


## Processing Function

Extracts the acoustic parameters from one saved oscillogram image.
`pixel_time` comes from the caller, not OCR.

In [ ]:
def frog_process(spectrogram_path, pixel_time):
    """Extract the acoustic parameters from one oscillogram image: ink →
    on/off signal → trim silence → classify gaps."""
    img = Image.open(str(spectrogram_path)).convert('L')
    spec_array = np.array(img)

    if spec_array.shape[1] == 0:
        raise ValueError('Empty image')

    ink = extract_outer_band_ink(spec_array)
    raw_signal_list, _ = detect_signal_list_adaptive(ink, pixel_time)

    signal_columns = np.where(raw_signal_list == 1)[0]
    if not len(signal_columns):
        raise ValueError('No signal columns detected')

    trim_start = max(0, signal_columns[0] - 2)
    trim_end = min(len(raw_signal_list), signal_columns[-1] + 3)

    leading_silence = trim_start * pixel_time
    trailing_silence = (len(raw_signal_list) - trim_end) * pixel_time

    signal_list = clean_signal_runs(
        raw_signal_list[trim_start:trim_end], pixel_time, fill_gap_size=0
    )
    if not len(signal_list) or signal_list.mean() == 0:
        raise ValueError('Empty signal after cleanup')

    time_buckets = [
        ('On' if value == 1 else 'Off', run_length * pixel_time)
        for value, run_length in rle(signal_list)
    ]

    on_durations = [duration for state, duration in time_buckets if state == 'On']
    if not on_durations:
        raise ValueError('No on-pulses detected')
    element_length = float(np.mean(on_durations))

    (
        inter_element_interval, inter_burst_interval, elements_per_burst,
        min_elements_per_burst, max_elements_per_burst,
    ) = classify_intervals(
        time_buckets, leading_silence, trailing_silence, element_length, pixel_time
    )

    return {
        'element_length': round(element_length, 4),
        'inter_element_interval': round(inter_element_interval, 4),
        'inter_burst_interval': round(inter_burst_interval, 4),
        'elements_per_burst': elements_per_burst,
        'min_elements_per_burst': min_elements_per_burst,
        'max_elements_per_burst': max_elements_per_burst,
    }


## Batch Run

For each clip: generate its oscillogram if it doesn't already exist, work out
`pixel_time` from the clip duration and image width, then run `frog_process`
and collect the result.

In [ ]:
rows = []

for _, clip_row in clips_ready.iterrows():
    wav_path = Path(clip_row['clipped_file'])
    clip_duration = float(clip_row['clip_duration_s'])
    genus = clip_row['genus']
    species = clip_row['species']

    spec_path = SPEC_DIR / (wav_path.stem + '_spectrogram.png')

    try:
        if not spec_path.exists():
            generate_spectrogram_image(wav_path, spec_path)

        # pixel_time comes from the saved image width and the known clip
        # duration, so we don't have to assume anything about DPI/figsize.
        img_width = Image.open(str(spec_path)).size[0]
        pixel_time = clip_duration / img_width

        result = frog_process(spec_path, pixel_time)
        status = 'ok'
        error = ''

    except Exception as e:
        result = {}
        status = 'error'
        error = str(e)

    rows.append({
        'genus': genus,
        'species': species,
        'file': wav_path.name,
        'status': status,
        'error': error,
        **result,
    })

output_csv = SPEC_DIR / 'frog_results.csv'
csv_columns = [
    'genus', 'species', 'file', 'status', 'error',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]
with open(output_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=csv_columns, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(rows)

successful = [r for r in rows if r['status'] == 'ok']
errors = [r for r in rows if r['status'] == 'error']
print(f'Processed {len(rows)} clips: {len(successful)} ok, {len(errors)} errors')
for r in errors:
    print(f"  ERROR {r['genus']} {r['species']} / {r['file']}: {r['error']}")


## Results

In [ ]:
df    = pd.read_csv(output_csv)
df_ok = df[df['status'] == 'ok'].copy()
print(f'{len(df_ok)} clips processed successfully')
df_ok[[
    'genus', 'species', 'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]].head(20)

## Build frog_final DataFrame

Merges the processing results with Xeno-Canto metadata (`frog_metadata.csv`):
Country, Location, Latitude, Longitude, Call Type. No Temperature--Xeno-Canto
doesn't report it consistently the way SINA does. `Amerana_draytonii` is left
blank here since its metadata can't be reliably matched back to individual
clips.

In [ ]:
# Derive File_ID from the WAV filename stem (e.g. 'Rana_temporaria_1' -> '1')
df_ok = df_ok.copy()
df_ok['File_ID'] = df_ok['file'].apply(
    lambda f: re.search(r'_(\d+)\.wav$', f).group(1)
    if re.search(r'_(\d+)\.wav$', f) else os.path.splitext(f)[0]
)

frog_final = df_ok[[
    'genus', 'species', 'File_ID',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]].rename(columns={
    'genus': 'Genus',
    'species': 'Species',
    'element_length': 'Element_Length',
    'inter_element_interval': 'Inter-Element_Interval',
    'inter_burst_interval': 'Inter-Burst_Interval',
    'elements_per_burst': 'Elements_Per_Burst',
    'min_elements_per_burst': 'Min_Elements_Per_Burst',
    'max_elements_per_burst': 'Max_Elements_Per_Burst',
})

# Xeno-Canto metadata (Country/Location/Lat/Lon/Call Type): a lookup keyed by
# (Genus, Species, audio_num). Amerana_draytonii is left blank--its on-disk
# file count doesn't match frog_df, so the audio_num join isn't trustworthy.
frog_metadata = pd.read_csv(AUDIO_DIR / 'frog_metadata.csv')
frog_final['audio_num'] = frog_final['File_ID'].astype(int)
frog_final = frog_final.merge(
    frog_metadata, on=['Genus', 'Species', 'audio_num'], how='left'
).drop(columns='audio_num')

print(f'{len(frog_final)} rows | {frog_final["Genus"].nunique()} genera | {frog_final["Species"].nunique()} species')
frog_final.head()


In [ ]:
%store frog_final

In [ ]:
frog_final.to_csv(Path.home() / 'Discrete_Signals' / 'frog_single_burst_type.csv', index=False)